# Bandcamp Artist Discog Scraper
Based on:
- dbeley's [bandcamp-library-scraper.py](https://github.com/dbeley/bandcamp-library-scraper/blob/main/bandcamp-library-scraper.py)
- nahnhh's [webcrawl2_movie details.ipynb](https://github.com/nahnhh/top-movies-2005-2025/blob/main/webcrawl2_movie%20details.ipynb)

Artist list used: `slushwave-bandcamp-links.txt` compiled in Slushwave Social Club Discord server.

`image-links.csv` only gets the smallest file size, the one ending with [_1x1_700.avif](https://f4.bcbits.com/img/a3705632330_1x1_700.avif)

In [1]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
BASE_DIR = Path.cwd()
LINKS_FILE = BASE_DIR / "slushwave-bandcamp-links.txt"
OUTPUT_FILE = BASE_DIR / "bc-albums-image-links.csv"
IMAGES_DIR = BASE_DIR / "images"

In [ ]:
import requests
from bs4 import BeautifulSoup

def parse_artist_name(raw_text: str) -> str:
    """Trim the 'by ' prefix used by Bandcamp artist labels without chopping the name."""
    text = raw_text.strip()
    if text.lower().startswith("by "):
        return text[3:].strip()
    if text.lower().startswith("par "):
        return text[4:].strip()
    return text

def big_image(image_link: str) -> str:
    """Convert a Bandcamp image link to the biggest version."""
    if image_link.endswith("_1x1_700.avif"):
        return image_link[:-13] + "_0"
    return image_link

### List of browers to rotate, spin them once a while

In [3]:
HEADERS_LIST = [
  {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:89.0) Gecko/20100101 Firefox/89.0",
    "X-Requested-With": "XMLHttpRequest"
  },
  {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.13; rv:110.0) Gecko/20100101 Firefox/110.0",
    "X-Requested-With": "XMLHttpRequest"
  },
  {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.75 Safari/537.36",
    "X-Requested-With": "XMLHttpRequest"
  },
  {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36 Edg/110.0.1587.50",
    "X-Requested-With": "XMLHttpRequest"
  },
  {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "X-Requested-With": "XMLHttpRequest"
  }
]

### Async webscraping

In [ ]:
import csv
import logging
import re
from datetime import datetime

def get_soup(url: str, session: requests.Session) -> BeautifulSoup:
	response = session.get(url, timeout=30)
	response.raise_for_status()
	return BeautifulSoup(response.content, "lxml")

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()


def extract_image_url(album_element) -> str:
	"""Extract the image URL from album element."""
	try:
		# Find the picture element within the album item
		picture = album_element.find("picture")
		if picture:
			# Look for source tags with the AVIF format
			source = picture.find("source")
			if source and source.get("srcset"):
				# srcset can contain multiple URLs, get the first one
				image_url = source.get("srcset").split()[0]
				return image_url
			# Fallback to img tag
			img = picture.find("img")
			if img and img.get("src"):
				return img.get("src")
	except Exception as e:
		print(f"Error extracting image URL: {e}")
	return "Error"


def download_image(session: requests.Session, image_url: str, filename: str) -> bool:
    """Download an image file from URL."""
    try:
        response = session.get(image_url, timeout=30)
        response.raise_for_status()
        with open(filename, "wb") as f:
            f.write(response.content)
        return True
    except Exception as e:
        print(f"Error downloading image from {image_url}: {e}")
    return False

def extract_discography(artist, session: requests.Session):
    soup_artist = get_soup(artist["url"] + "music", session)
    list_albums = []
    try:
        for item in (
            soup_artist.find("div", {"class": "leftMiddleColumns"})
            .find("ol", {"id": "music-grid"})
            .find_all("li", {"class": "music-grid-item"})
        ):
            album_element = item.find("p", {"class": "title"})
            album_name = album_element.text.strip()
            alternate_artist = None
            if alternate_element := album_element.find(
                "span", {"class": "artist-override"}
            ):
                logger.debug(
                    f"Trying to parse alternate artist name in {album_element}"
                )
                album_name = album_name.split("\n")[0]
                alternate_artist = alternate_element.text.strip()
                logger.debug(f"Found {alternate_artist}")
            url = item.find("a")["href"]
            
            # Extract and download image
            image_url = extract_image_url(item)
            image_filename = None
            if image_url:
                image_filename = f"{artist['name']}_{album_name.replace('/', '_')}.avif"
                image_path = IMAGES_DIR / image_filename
                IMAGES_DIR.mkdir(exist_ok=True)
                if download_image(session, big_image(image_url), image_path):
                    image_filename = image_filename
            
            list_albums.append(
                {
                    "artist": artist["name"],
                    "alternate_artist": alternate_artist,
                    "name": album_name,
                    "url": url if url.startswith("https://") else artist["url"] + url,
                    "image": image_filename,
                }
            )
    except Exception as e:
        logger.warning(
            f"Couldn't extract informations for artist {artist['name']}: {e}"
        )
        return []
    
    return list_albums


def export_to_csv(list_items, filename: str):
    with open(filename, "w", encoding="utf8", newline="") as output_file:
        fc = csv.DictWriter(
            output_file, fieldnames=sorted(set().union(*(d.keys() for d in list_items)))
        )
        fc.writeheader()
        fc.writerows(list_items)

In [5]:
import asyncio
import aiohttp
from bs4 import BeautifulSoup
from tqdm.asyncio import tqdm

max_concurrency = 10
sem = asyncio.Semaphore(max_concurrency)
timeout_urls = []

async def scrape_movie_details_async(session, header, url):
  """
  Async scraping of movie details.
  """
  async with sem:
    try:
      async with session.get(url, headers=header, timeout=25) as r:
        status = r.status
        if status != 200:
          print(f"HTTP {status} for {url}")
          return None
        html = await r.text()
        soup = BeautifulSoup(html, 'html.parser')
        
        # Use the shared parsing logic
        data = parse_movie_details(soup, url)
        # Treat dicts with only 'link' as invalid
        if isinstance(data, dict) and len(data) <= 1:
          print(f"Parsed no data for {url}")
          return None
        return data
    
    except asyncio.TimeoutError:
        print(f"Timeout error for {url}")
        return None
    except Exception as e:
        print(f"Error for {url}: {e}")
        return None

async def scrape_batch(session, header, urls):
  """Async scraping, continued - callable function."""
  if session is None:
    async with aiohttp.ClientSession() as session:
      tasks = [scrape_movie_details_async(session, header, url) for url in urls]
      return await tqdm.gather(*tasks)
  else:
    tasks = [scrape_movie_details_async(session, header, url) for url in urls]
    return await tqdm.gather(*tasks)